## This section focuses on exploring and implementing advanced concepts and techniques in information retrieval. The primary objectives are to build Retrieval Augumentation Generation, and learn about Language Models

## Author: Nam Dang

## Setting up the libraries and the environment

In [ ]:
%pip install langchain_community transformers langchain_text_splitters sentence_transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
%pip install pandas accelerate

## Data Preprocessing and Model Selection

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving food_recipes.csv to food_recipes.csv


In [ ]:
import pandas as pd

df = pd.read_csv("food_recipes.csv")

df = df.head(1000)

df["text"] = "Recipe: " + df["title"] + ". Directions: " + df["directions"]

print(f"Loaded {len(df)} recipes")
print(df["text"].iloc[0])

Loaded 1000 recipes
Recipe: No-Bake Nut Cookies. Directions: In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine., Stir over medium heat until mixture bubbles all over top., Boil and stir 5 minutes more. Take off heat., Stir in vanilla and cereal; mix well., Using 2 teaspoons, drop and shape into 30 clusters on wax paper., Let stand until firm, about 30 minutes.


In [ ]:
from transformers import AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenized_texts = tokenizer(
    df["text"].tolist(), truncation=True, max_length=512, padding=False
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Split each recipe text into chunks
document_chunks = []
for text in df["text"].tolist():
    chunks = text_splitter.split_text(text)
    document_chunks.extend(chunks)

print(f"Created {len(document_chunks)} chunks")

print("\nExample:")
print(document_chunks[1])

Created 1072 chunks

Example:
Recipe: Jewell Ball'S Chicken. Directions: Place chipped beef on bottom of baking dish., Place chicken on top of beef., Mix soup and cream together; pour over chicken. Bake, uncovered, at 275° for 3 hours.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Initialize the embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

# embeddings
embeddings = embedding_model.encode(document_chunks)
print(f"Created {len(embeddings)} embeddings with shape {embeddings.shape}")

# FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

# Create a mapping from index position to document chunk for retrieval
index_to_chunk = {i: chunk for i, chunk in enumerate(document_chunks)}

print(f"Created FAISS index with {index.ntotal} vectors")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Created 1072 embeddings with shape (1072, 384)
Created FAISS index with 1072 vectors


## Implementing RAG using LangChain for different queries

RAG pipeline:

- Documents: The source information such as recipes, articles, PDFs is collected

- Chunks: Documents are split into smaller chunks to be retrieved

- Embeddings: Each chunk is converted into a numerical vector representation (embedding) that captures its meaning

- Indexing: The embeddings are stored in an vector database

- Query: question from the user is encoded into vector representation

- Retrieval: The encoded query is compared with the stored document embeddings to check if they match (semantic similarity)

- Generation: The LLM uses the retrieved information to generate an answer to the user

I will use the TinyLlama/TinyLlama-1.1B-Chat-v1.0 pretrained language model. It is a small model that works on CPU, which makes it practical for this assignment because we don't need GPU. It also understands prompts such as <|system|>, <|user|>, <|assistant|>. They will be useful for answering questions about recipes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load LLM
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)

print(f"Successfully loaded {model_id}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Successfully loaded TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [ ]:
def rag_response(query, index, embedding_model, llm_model, llm_tokenizer, index_to_chunk_map, top_k=3):
    """
    Generate a response using the RAG pattern.

    Args:
        query: The user's question
        index: FAISS index
        embedding_model: Model to create embeddings
        llm_model: Language model for generation
        llm_tokenizer: Tokenizer for the language model
        index_to_chunk_map: Mapping from index positions to recipe chunks
        top_k: Number of chunks to retrieve

    Returns:
        response: The generated response
        sources: The retrieved chunks used as context
    """
    # Step 1: Convert query to embedding
    query_embedding = embedding_model.encode([query])
    query_embedding = query_embedding.astype(np.float32)

    # Step 2: Search for similar chunks
    distances, indices = index.search(query_embedding, top_k)

    # Step 3: Retrieve the actual chunks
    retrieved_chunks = [index_to_chunk_map[idx] for idx in indices[0]]

    # Create context from retrieved chunks
    context = "\n\n".join(retrieved_chunks)

    # Step 4: Create prompt for the LLM (TinyLlama format)
    prompt = f"""<|system|>
You are a helpful cooking assistant. Answer the question based only on the provided context.
If you don't know the answer based on the context, write "Not enough information!"

Context:
{context}
<|user|>
{query}
<|assistant|>"""

    # Step 5: Generate response from LLM
    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)

    with torch.no_grad():
        output = llm_model.generate(
            input_ids=input_ids,
            max_new_tokens=400,
            temperature=0.7,
            top_p=0.95,
            do_sample=True
        )

    # Decode the output
    generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract the assistant's response (remove the prompt)
    response = generated_text.split("<|assistant|>")[-1].strip()

    return response, retrieved_chunks

In [ ]:
test_queries = [
    "How should I cook Chinese chicken salad?",
    "What ingredients should I use to make fruit cake?"
]

results = []
for query in test_queries:
    response, sources = rag_response(
        query=query,
        index=index,
        embedding_model=embedding_model,
        llm_model=model,
        llm_tokenizer=tokenizer,
        index_to_chunk_map=index_to_chunk,
        top_k=2
    )
    results.append((query, response, sources))

    print(f"\nQuery: {query}")
    print(f"Answer: {response}")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: How should I cook Chinese chicken salad?
Answer: As per the provided context, the recipe does not specify how to cook Chinese chicken salad. You can safely assume that you need to cook saifon, a little at a time, in 1-inch of hot oil, until noodles "explode" and turn other side. You should then remove and drain the noodles and thinly slice them, and refrigerate the salad. You can also toast sesame seeds until lightly browned.

Query: What ingredients should I use to make fruit cake?
Answer: To make fruit cake, you need:
- eggs
- sugar
- flour (for the batter)
- chocolate chips or nuts (optional)
- cherries (optional)
- baking powder
- salt
- oil
- water
- peaches (for the filling)
- vanilla extract
- cake mix

You can substitute some or all of these ingredients with your own favorite flavors and ingredients.


In [ ]:
for query, response, sources in results:
    print(f"\nQuery: {query}")

    print(f"Answer: {response}")

    print("Retrieved information:")

    for i, chunk in enumerate(sources):
        print(f"  Source {i+1}: {chunk}")


Query: How should I cook Chinese chicken salad?
Answer: As per the provided context, the recipe does not specify how to cook Chinese chicken salad. You can safely assume that you need to cook saifon, a little at a time, in 1-inch of hot oil, until noodles "explode" and turn other side. You should then remove and drain the noodles and thinly slice them, and refrigerate the salad. You can also toast sesame seeds until lightly browned.
Retrieved information:
  Source 1: Recipe: Chinese Chicken Salad. Directions: Cook saifon, a little at a time, in 1-inch of hot oil., Cook until noodles "explode.", Turn and cook other side., Remove and drain on paper towel., Thinly slice the lettuce, cucumbers, celery, bell pepper, scallions and radishes., (Please improvise on salad if you wish.), Put in large bowl and refrigerate., Toast sesame seeds until lightly browned.
  Source 2: Recipe: Chicken Salad. Directions: Combine all ingredients; chill.

Query: What ingredients should I use to make fruit ca

Analysis: The retrieved information match the recipe titles mentioned in each query. For example, in the second question, I mention fruit cake, and the sources show "Fruit Cake" and "Peachy Cake". This shows that the retriever is finding the right recipes.

The first question's response is also drawn from the relevant sources named "Chicken Salad" in the recipe name. However, the response also stated that "the recipe does not specify how to cook Chinese chicken salad", but then proceed to gie instructions. This is inconsistent.

But overall, the generated answers are based exactly on the sources without making up any information. This shows the RAG pipeline is actually working for this food recipes dataset.

## Modify and evaluate the different components of RAG

In [ ]:
def expand_query(original_query, llm_model, llm_tokenizer):
    """
    Generate multiple search queries from an original query to improve retrieval

    Args:
        original_query: The user's original question
        llm_model: The language model for generating variations
        llm_tokenizer: Tokenizer for the language model

    Returns:
        List of query variations including the original
    """
    prompt = f"""<|system|>
You are a helpful assistant. Generate two alternative versions of the given search query.
The goal is to create variations that might help retrieve relevant information.
Only list the alternative queries, one per line. Do not include any explanations, numbering, or other text.
<|user|>
Generate alternative versions of this search query: "{original_query}"
<|assistant|>"""

    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)

    with torch.no_grad():
        output = llm_model.generate(
            input_ids=input_ids,
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True
        )

    generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=True)
    response_part = generated_text.split("<|assistant|>")[-1].strip()

    variations = [line.strip() for line in response_part.split('\n') if line.strip()]
    if not variations:
        variations = [original_query]

    all_queries = [original_query] + variations
    return list(dict.fromkeys(all_queries))

In [ ]:
def rag_response_expansion(query, index, embedding_model, llm_model, llm_tokenizer, index_to_chunk_map, top_k=3):

    expanded_queries = expand_query(query, llm_model, llm_tokenizer)

    # Track best score for each retrieved chunk
    all_scores = {}
    for q in expanded_queries:
        q_embedding = embedding_model.encode([q]).astype(np.float32)
        distances, indices = index.search(q_embedding, top_k)

        for idx, dist in zip(indices[0], distances[0]):
            score = 1.0 / (1.0 + dist)
            if idx in all_scores:
                all_scores[idx] = max(all_scores[idx], score)
            else:
                all_scores[idx] = score

    # Get top chunks based on scores
    top_indices = sorted(all_scores.keys(), key=lambda idx: all_scores[idx], reverse=True)[:top_k]
    retrieved_chunks = [index_to_chunk_map[idx] for idx in top_indices]

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""<|system|>
You are a helpful cooking assistant. Answer the question using only the recipe information provided in the context.
Do not add information that is not supported by the context.
If the context does not contain enough information, write "Not enough information!"

Context:
{context}

<|user|>
{query}
<|assistant|>"""

    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            input_ids=input_ids,
            max_new_tokens=400,
            temperature=0.7,
            top_p=0.95,
            do_sample=True
        )

    generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=True)
    response = generated_text.split("<|assistant|>")[-1].strip()

    return response, retrieved_chunks, expanded_queries

In [ ]:
comparison_query = "How do I make a chicken pot pie?"

# Standard retrieval
standard_response, standard_sources = rag_response(
    query=comparison_query,
    index=index,
    embedding_model=embedding_model,
    llm_model=model,
    llm_tokenizer=tokenizer,
    index_to_chunk_map=index_to_chunk,
    top_k=2
)

# Query expansion retrieval
expanded_response, expanded_sources, expanded_queries = rag_response_expansion(
    query=comparison_query,
    index=index,
    embedding_model=embedding_model,
    llm_model=model,
    llm_tokenizer=tokenizer,
    index_to_chunk_map=index_to_chunk,
    top_k=2
)

print(f"\n----- Comparison Query -----\n{comparison_query}")

print(f"\nExpanded queries: {expanded_queries}")

print(f"\n----- Standard Retrieval -----\n{standard_response}")

print(f"\n----- Query Expansion -----\n{expanded_response}")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



----- Comparison Query -----
How do I make a chicken pot pie?

Expanded queries: ['How do I make a chicken pot pie?', '1. How do I make a chicken pot pie?', '2. How do I make a delicious chicken pot pie?', '3. How do I make a flavorful chicken pot pie?', '4. How do I make a delicious and easy chicken pot pie?', '5. How do I make a perfect chicken pot pie?', '6. How do I make a mouthwatering chicken pot pie?', '7. How do I make a del']

----- Standard Retrieval -----
To make a chicken pot pie, you need to follow these steps:
1. Grease a 2-quart baking dish with cooking spray or butter.
2. In a shallow dish, combine vegetables (such as celery, onion, and carrot), chicken, broth, soup, salt, and pepper.
3. In a separate bowl, combine flour, milk, and margarine. Mix until smooth.
4. Pour the wet mixture over the chicken mixture in the shallow dish.
5. Bake in a preheated 400°F oven for 40-50 minutes or until the crust is golden brown.

Note: You can adjust the vegetables and seasonings to

In [ ]:
def rag_response_improved(query, index, embedding_model, llm_model, llm_tokenizer, index_to_chunk_map, top_k=3):

    query_embedding = embedding_model.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, top_k)
    retrieved_chunks = [index_to_chunk_map[idx] for idx in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # Modified prompt
    prompt = f"""<|system|>
You are a professional cooking assistant. Answer the question using only the recipes provided in the context.
Do not add, assume, or make up any ingredients, quantities, cooking times, temperatures, or instructions.
Give the answer as concise and numbered steps.
If the context does not contain enough information, write "Not enough information!"

Context:
{context}

<|user|>
{query}
<|assistant|>"""

    input_ids = llm_tokenizer(prompt, return_tensors="pt").input_ids.to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            input_ids=input_ids,
            max_new_tokens=400,
            temperature=0.7,
            top_p=0.95,
            do_sample=True
        )

    generated_text = llm_tokenizer.decode(output[0], skip_special_tokens=True)
    response = generated_text.split("<|assistant|>")[-1].strip()

    return response, retrieved_chunks

In [ ]:
# Compare original prompt vs improved prompt
improved_response, improved_sources = rag_response_improved(
    query=comparison_query,
    index=index,
    embedding_model=embedding_model,
    llm_model=model,
    llm_tokenizer=tokenizer,
    index_to_chunk_map=index_to_chunk,
    top_k=2
)

print(f"\n----- Comparison Query -----\n{comparison_query}")

print(f"\n----- Original Prompt -----\n{standard_response}")

print(f"\n----- Improved Prompt -----\n{improved_response}")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



----- Comparison Query -----
How do I make a chicken pot pie?

----- Original Prompt -----
To make a chicken pot pie, you need to follow these steps:
1. Grease a 2-quart baking dish with cooking spray or butter.
2. In a shallow dish, combine vegetables (such as celery, onion, and carrot), chicken, broth, soup, salt, and pepper.
3. In a separate bowl, combine flour, milk, and margarine. Mix until smooth.
4. Pour the wet mixture over the chicken mixture in the shallow dish.
5. Bake in a preheated 400°F oven for 40-50 minutes or until the crust is golden brown.

Note: You can adjust the vegetables and seasonings to your preference. Enjoy your homemade chicken pot pie!

----- Improved Prompt -----
Here's a basic recipe for making a chicken pot pie:

Ingredients:
- 2 cups cooked chicken breast, shredded
- 1 cup frozen peas, thawed
- 1 cup frozen corn, thawed
- 1 cup frozen green beans, thawed
- 1 large onion, chopped
- 4 cloves garlic, minced
- 3 cups chicken broth
- 1 cup low-fat milk
- 1

In [ ]:
for k in [1, 3, 5]:
    response_k, sources_k = rag_response(
        query=comparison_query,
        index=index,
        embedding_model=embedding_model,
        llm_model=model,
        llm_tokenizer=tokenizer,
        index_to_chunk_map=index_to_chunk,
        top_k=k
    )

    print(f"\n----- top_k = {k} -----")

    print(f"Number of retrieved chunks: {len(sources_k)}")

    print(f"Response: {response_k}\n")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



----- top_k = 1 -----
Number of retrieved chunks: 1
Response: To make a chicken pot pie, you need to follow these steps:

1. Preheat oven to 375°F (190°C).
2. In a mixing bowl, combine the cooked chicken, vegetables, and sauce.
3. Add the eggs, milk, salt, and pepper to the mixture. Mix well.
4. Pour the mixture into a 9 x 13-inch baking dish.
5. Bake for 45-50 minutes, or until the edges are golden brown.
6. Let the chicken pot pie cool for a few minutes before serving. Enjoy!



[transformers] Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



----- top_k = 3 -----
Number of retrieved chunks: 3
Response: To make a chicken pot pie, you need to follow the instructions for the recipe provided. Please make sure to follow the recipe exactly as written or you may end up with an incorrect result. If you are still unsure, it would be best to consult the recipe and ask for guidance from a professional chef.


----- top_k = 5 -----
Number of retrieved chunks: 5
Response: To make a chicken pot pie, you will need:

1. Preheat the oven to 375°F (190°C)
2. Grease a 9 x 13-inch baking dish
3. Mix together a vegetable broth, chicken, soup, salt, and pepper
4. Mix in flour, milk, and margarine until smooth
5. Pour the mixture over the chicken and vegetables in the baking dish
6. Bake in the preheated oven for 40-50 minutes
7. Spread Veg-All over the top of the chicken mixture
8. Bake for another 10-15 minutes or until the crust is golden brown.



**Analysis**:

- **top_k=1:** The response was clear with numbered steps and specific ingredients.


- **top_k=3:** The model did not give any steps at all. It just said to follow the recipe without listing anything. This could be because the 3 retrieved chunks were mixed or unclear, so the model got confused and gave a vague answer.


- **top_k=5:** The response was the most detailed and closest to a real recipe. It is not far better than top_k=1, but more detailed and specific. The steps are also arranged more logically.

In [ ]:
print(f"\n----- Comparison Query -----\n{comparison_query}")

print("\n----- Standard retrieval -----")
print(standard_response)

print("\n----- Query expansion retrieval -----")
print(expanded_response)

print("\n----- Improved prompt template -----")
print(improved_response)


----- Comparison Query -----
How do I make a chicken pot pie?

----- Standard retrieval -----
To make a chicken pot pie, you need to follow these steps:
1. Grease a 2-quart baking dish with cooking spray or butter.
2. In a shallow dish, combine vegetables (such as celery, onion, and carrot), chicken, broth, soup, salt, and pepper.
3. In a separate bowl, combine flour, milk, and margarine. Mix until smooth.
4. Pour the wet mixture over the chicken mixture in the shallow dish.
5. Bake in a preheated 400°F oven for 40-50 minutes or until the crust is golden brown.

Note: You can adjust the vegetables and seasonings to your preference. Enjoy your homemade chicken pot pie!

----- Query expansion retrieval -----
To make a chicken pot pie, follow the steps provided in the context:

1. Mix together Veg-All and 3 cans of soup in a large bowl.
2. Spread this mixture on top of the chicken.
3. In the same bowl, mix melted butter, milk, and flour.
4. Pour this mixture over the soup and chicken mix

**Analysis:**
- **Standard**: The model gave a clear, complete recipe with actual steps and some ingredients. The answer stayed short and matched a real recipe closely.

- **Query Expansion**: The model also gave a full, clear recipe, but pulled from a different chicken pot pie version. This shows query expansion can retrieve a different but still matching recipe.

- **Improved**: Gave the longest and most detailed response, with a full ingredient list and numbered instructions. But looking closely, some ingredients are listed twice. This means the improved prompt made the answer look more professional and complete, but the content is still not perfect.

---

**Conclusion**: All three versions gave actual chicken pot pie recipes from the dataset, so retrieval was working correctly in every case. Standard retrieval gave the shortest and cleanest response. Query expansion gave a different but still correct recipe. The improved prompt gave the most detailed and structured answer but also showed some imperfection. This shows that a more detailed prompt can improve response structure but may also produce small errors.


## Selecting and implementing a pretrained model for a new task

I choose text classification as the new task. I think it is helpful to classify each recipe into a food category such as dessert, main course, drink, etc. For example, users could easily find all the dessert recipes without having to go through the whole dataset.

It shows that a language model can understand the meaning of the recipes and put them to corresponding categories.

Reference: https://huggingface.co/facebook/bart-large-mnli

I choose facebook/bart-large-mnli for this text classification task. It is trained using Supervised Fine-Tuning (SFT). This model is good for zero-shot text classification. I do not need to train the model on my dataset. Instead, I can provide custom labels and the model will determine which label best matches each recipe.

In [ ]:
# Reference: https://huggingface.co/tasks/zero-shot-classification

from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

print("Zero-shot classification pipeline loaded")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Zero-shot classification pipeline loaded


In [ ]:
type_of_food = ["Appetizer", "Main Dish", "Dessert", "Drink"]

# Test with 5 recipes
test_recipes = df["text"].iloc[:5].tolist()

for i, recipe_text in enumerate(test_recipes):
    result = classifier(recipe_text, type_of_food)

    print(f"\nRecipe {i+1}: {df['title'].iloc[i]}")

    print(f"Predicted category: {result['labels'][0]} (score: {result['scores'][0]:.2f})")

    print("\nAll scores:")
    for label, score in zip(result['labels'], result['scores']):
        print(f"  {label}: {score:.4f}")


Recipe 1: No-Bake Nut Cookies
Predicted category: Dessert (score: 0.36)

All scores:
  Dessert: 0.3571
  Drink: 0.2740
  Main Dish: 0.2145
  Appetizer: 0.1544

Recipe 2: Jewell Ball'S Chicken
Predicted category: Main Dish (score: 0.37)

All scores:
  Main Dish: 0.3709
  Drink: 0.3559
  Appetizer: 0.2189
  Dessert: 0.0543

Recipe 3: Creamy Corn
Predicted category: Main Dish (score: 0.41)

All scores:
  Main Dish: 0.4068
  Drink: 0.2751
  Appetizer: 0.2352
  Dessert: 0.0828

Recipe 4: Chicken Funny
Predicted category: Main Dish (score: 0.44)

All scores:
  Main Dish: 0.4423
  Appetizer: 0.2803
  Drink: 0.1973
  Dessert: 0.0802

Recipe 5: Reeses Cups(Candy)  
Predicted category: Dessert (score: 0.29)

All scores:
  Dessert: 0.2901
  Drink: 0.2839
  Main Dish: 0.2163
  Appetizer: 0.2098


Looking at the 5 example outputs above, the predicted categories actually make sense because the model correctly identifies which category a recipe belong to. This shows that this model is working well for the text classification task for my food recipes dataset